In [1]:
# ============================================================
# Dataset: Kerala PCOS Clinical Dataset (n=541)
# ============================================================

import pandas as pd
import numpy as np

# ============================================================
# 1. LOAD DATA
# ============================================================

RAW_PATH = '/kaggle/input/datasets/prasoonkottarathil/polycystic-ovary-syndrome-pcos/PCOS_data_without_infertility.xlsx'

df = pd.read_excel(RAW_PATH, sheet_name='Full_new')

print("Raw shape:", df.shape)
print("Class distribution (raw):")
print(df['PCOS (Y/N)'].value_counts())


# ============================================================
# 2. DROP ADMINISTRATIVE / JUNK COLUMNS
# ============================================================

drop_cols = ['Sl. No', 'Patient File No.', 'Unnamed: 44']
df.drop(columns=drop_cols, inplace=True)

print("\nAfter dropping junk columns:", df.shape)


# ============================================================
# 3. RENAME COLUMNS — clean, Python-safe names
# ============================================================

rename_map = {
    'PCOS (Y/N)'              : 'PCOS',
    ' Age (yrs)'              : 'Age',
    'Weight (Kg)'             : 'Weight_kg',
    'Height(Cm) '             : 'Height_cm',
    'BMI'                     : 'BMI',
    'Blood Group'             : 'Blood_Group',
    'Pulse rate(bpm) '        : 'Pulse_bpm',
    'RR (breaths/min)'        : 'RR_bpm',
    'Hb(g/dl)'                : 'Hb_gdl',
    'Cycle(R/I)'              : 'Cycle_RI',
    'Cycle length(days)'      : 'Cycle_length_days',
    'Marraige Status (Yrs)'   : 'Marriage_yrs',
    'Pregnant(Y/N)'           : 'Pregnant',
    'No. of aborptions'       : 'Abortions',
    '  I   beta-HCG(mIU/mL)' : 'BetaHCG_I',
    'II    beta-HCG(mIU/mL)'  : 'BetaHCG_II',
    'FSH(mIU/mL)'             : 'FSH',
    'LH(mIU/mL)'              : 'LH',
    'FSH/LH'                  : 'FSH_LH_ratio',
    'Hip(inch)'               : 'Hip_inch',
    'Waist(inch)'             : 'Waist_inch',
    'Waist:Hip Ratio'         : 'WHR',
    'TSH (mIU/L)'             : 'TSH',
    'AMH(ng/mL)'              : 'AMH',
    'PRL(ng/mL)'              : 'PRL',
    'Vit D3 (ng/mL)'          : 'VitD3',
    'PRG(ng/mL)'              : 'PRG',
    'RBS(mg/dl)'              : 'RBS',
    'Weight gain(Y/N)'        : 'WeightGain',
    'hair growth(Y/N)'        : 'HairGrowth',
    'Skin darkening (Y/N)'    : 'SkinDarkening',
    'Hair loss(Y/N)'          : 'HairLoss',
    'Pimples(Y/N)'            : 'Pimples',
    'Fast food (Y/N)'         : 'FastFood',
    'Reg.Exercise(Y/N)'       : 'RegExercise',
    'BP _Systolic (mmHg)'     : 'BP_Systolic',
    'BP _Diastolic (mmHg)'    : 'BP_Diastolic',
    'Follicle No. (L)'        : 'Follicle_L',
    'Follicle No. (R)'        : 'Follicle_R',
    'Avg. F size (L) (mm)'    : 'FSize_L',
    'Avg. F size (R) (mm)'    : 'FSize_R',
    'Endometrium (mm)'        : 'Endometrium',
}

df.rename(columns=rename_map, inplace=True)
print("\nColumns after renaming:")
print(list(df.columns))


# ============================================================
# 4. FIX DTYPE ISSUES — AMH and BetaHCG_II stored as object
# ============================================================

# AMH row 305 contains string 'a' — coerce to NaN, will impute below
df['AMH'] = pd.to_numeric(df['AMH'], errors='coerce')

# BetaHCG_II row 123 contains '1.99.' (trailing period) — fix typo
df['BetaHCG_II'] = df['BetaHCG_II'].replace('1.99.', '1.99')
df['BetaHCG_II'] = pd.to_numeric(df['BetaHCG_II'], errors='coerce')

print("\nDtype fixes applied.")
print(f"  AMH    — missing after coercion: {df['AMH'].isnull().sum()}")
print(f"  BetaHCG_II — missing after coercion: {df['BetaHCG_II'].isnull().sum()}")


# ============================================================
# 5. RECODE CYCLE_RI — 2=Regular → 0, 4/5=Irregular → 1
# ============================================================

# Original encoding: 2 = regular, 4 = irregular, 5 = one entry (treated as irregular)
df['Cycle_RI'] = df['Cycle_RI'].map({2: 0, 4: 1, 5: 1})

print("\nCycle_RI recoded:")
print(df['Cycle_RI'].value_counts())


# ============================================================
# 6. MARK PHYSICALLY IMPOSSIBLE VALUES AS NaN. These are 
# data entry errors, not biological extremes. Justification 
# documented per variable.
# ============================================================

# FSH = 5052 mIU/mL — impossible. Post-menopausal ceiling ~150 mIU/mL.
n = (df['FSH'] > 500).sum()
df.loc[df['FSH'] > 500, 'FSH'] = np.nan
print(f"\nFSH > 500 → NaN: {n} row(s)")

# LH = 2018 mIU/mL — impossible. Clinical ceiling even in LH surge ~200 mIU/mL.
n = (df['LH'] > 500).sum()
df.loc[df['LH'] > 500, 'LH'] = np.nan
print(f"LH > 500 → NaN: {n} row(s)")

# VitD3 = 6014 and 5418 ng/mL — impossible. Toxicity threshold ~150 ng/mL.
n = (df['VitD3'] > 200).sum()
df.loc[df['VitD3'] > 200, 'VitD3'] = np.nan
print(f"VitD3 > 200 → NaN: {n} row(s)")

# BP_Systolic = 12 mmHg — impossible in a living, conscious person.
n = (df['BP_Systolic'] < 50).sum()
df.loc[df['BP_Systolic'] < 50, 'BP_Systolic'] = np.nan
print(f"BP_Systolic < 50 → NaN: {n} row(s)")

# Pulse = 13 bpm — impossible without cardiac arrest.
n = (df['Pulse_bpm'] < 25).sum()
df.loc[df['Pulse_bpm'] < 25, 'Pulse_bpm'] = np.nan
print(f"Pulse_bpm < 25 → NaN: {n} row(s)")

# Recalculate FSH_LH_ratio since FSH and LH may now have new NaNs
# Will be handled after imputation below.


# ============================================================
# 7. MISSING VALUE AUDIT — before imputation
# ============================================================

missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("\nMissing values before imputation:")
print(missing)


# ============================================================
# 8. GROUP-STRATIFIED MEDIAN IMPUTATION
#    Impute within PCOS=0 and PCOS=1 groups separately.
#    Rationale: biomarker distributions differ between groups.
#    Cross-group imputation would introduce systematic bias.
# ============================================================

cols_to_impute = missing.index.tolist()

for col in cols_to_impute:
    for pcos_val in [0, 1]:
        group_mask  = df['PCOS'] == pcos_val
        missing_mask = df[col].isnull() & group_mask
        if missing_mask.sum() == 0:
            continue
        fill_value = df.loc[group_mask, col].median()
        df.loc[missing_mask, col] = fill_value
        print(f"  {col} (PCOS={pcos_val}): "
              f"{missing_mask.sum()} value(s) → group median {fill_value:.4f}")

print(f"\nMissing values after imputation: {df.isnull().sum().sum()}")


# ============================================================
# 9. RECOMPUTE FSH_LH_ratio AND ADD LH_FSH_ratio
#    FSH_LH_ratio: FSH ÷ LH  (as stored in original dataset)
#    LH_FSH_ratio: LH ÷ FSH  (clinically standard direction)
#    Both kept — FSH_LH_ratio for continuity, LH_FSH_ratio for
#    direct comparison with literature (threshold LH/FSH > 2).
# ============================================================

# Recompute FSH_LH_ratio now that FSH and LH errors are fixed
df['FSH_LH_ratio'] = df['FSH'] / df['LH'].replace(0, np.nan)

# Add clinically standard direction
df['LH_FSH_ratio'] = df['LH'] / df['FSH'].replace(0, np.nan)

# Any NaN from division (LH or FSH = 0) → group median
for ratio_col in ['FSH_LH_ratio', 'LH_FSH_ratio']:
    for pcos_val in [0, 1]:
        group_mask   = df['PCOS'] == pcos_val
        missing_mask = df[ratio_col].isnull() & group_mask
        if missing_mask.sum() > 0:
            fill_value = df.loc[group_mask, ratio_col].median()
            df.loc[missing_mask, ratio_col] = fill_value

print("\nRatio columns recomputed.")
print(f"  FSH_LH_ratio  — PCOS=0 median: {df[df['PCOS']==0]['FSH_LH_ratio'].median():.3f}"
      f"  |  PCOS=1 median: {df[df['PCOS']==1]['FSH_LH_ratio'].median():.3f}")
print(f"  LH_FSH_ratio  — PCOS=0 median: {df[df['PCOS']==0]['LH_FSH_ratio'].median():.3f}"
      f"  |  PCOS=1 median: {df[df['PCOS']==1]['LH_FSH_ratio'].median():.3f}")


# ============================================================
# 10. ONE-HOT ENCODE BLOOD GROUP
#     Blood_Group stores integer codes (11–18) for 8 blood types.
#     No ordinal meaning — must be one-hot encoded.
#     drop_first=True removes one dummy to avoid multicollinearity.
# ============================================================

bg_dummies = pd.get_dummies(df['Blood_Group'], prefix='BG', drop_first=True)
df = pd.concat([df.drop(columns=['Blood_Group']), bg_dummies], axis=1)

print("\nBlood Group one-hot encoded.")
print(f"  Dummy columns added: {list(bg_dummies.columns)}")


# ============================================================
# 11. KEY BIOMARKER INTEGRITY CHECK
#     Verify the 4 priority features survived preprocessing
#     without degradation. No values should be missing.
#     Distributions should be consistent with clinical literature.
# ============================================================

priority = ['AMH', 'LH', 'FSH', 'LH_FSH_ratio', 'Follicle_L', 'Follicle_R']

print("\n=== KEY BIOMARKER INTEGRITY CHECK ===")
print(f"{'Feature':<20} {'Missing':>8} {'Min':>8} {'Median':>8} "
      f"{'Max':>10} {'PCOS=0 med':>12} {'PCOS=1 med':>12}")
print("-" * 82)
for feat in priority:
    m   = df[feat].isnull().sum()
    mn  = df[feat].min()
    med = df[feat].median()
    mx  = df[feat].max()
    m0  = df[df['PCOS']==0][feat].median()
    m1  = df[df['PCOS']==1][feat].median()
    flag = "  ← EXPECTED HIGHER IN PCOS" if m1 > m0 else ""
    print(f"  {feat:<18} {m:>8} {mn:>8.3f} {med:>8.3f} {mx:>10.3f} "
          f"{m0:>12.3f} {m1:>12.3f}{flag}")


# ============================================================
# 12. FINAL AUDIT
# ============================================================

print("\n=== FINAL DATASET SUMMARY ===")
print(f"  Shape         : {df.shape}")
print(f"  Missing values: {df.isnull().sum().sum()}")
print(f"  PCOS=0        : {(df['PCOS']==0).sum()}")
print(f"  PCOS=1        : {(df['PCOS']==1).sum()}")
print(f"\nAll columns ({df.shape[1]}):")
print(list(df.columns))


# ============================================================
# 13. SAVE CLEANED DATASET
#     Save to Kaggle output directory.
#     Upload this as a new Kaggle dataset after running.
#     All downstream notebooks (graph construction, models)
#     attach this versioned output — never the raw file.
# ============================================================

OUTPUT_PATH = '/kaggle/working/pcos_cleaned.csv'
df.to_csv(OUTPUT_PATH, index=False)
print(f"\nCleaned dataset saved to: {OUTPUT_PATH}")
print("Upload this file as a new Kaggle dataset before proceeding to Phase 4.")

Raw shape: (541, 45)
Class distribution (raw):
PCOS (Y/N)
0    364
1    177
Name: count, dtype: int64

After dropping junk columns: (541, 42)

Columns after renaming:
['PCOS', 'Age', 'Weight_kg', 'Height_cm', 'BMI', 'Blood_Group', 'Pulse_bpm', 'RR_bpm', 'Hb_gdl', 'Cycle_RI', 'Cycle_length_days', 'Marriage_yrs', 'Pregnant', 'Abortions', 'BetaHCG_I', 'BetaHCG_II', 'FSH', 'LH', 'FSH_LH_ratio', 'Hip_inch', 'Waist_inch', 'WHR', 'TSH', 'AMH', 'PRL', 'VitD3', 'PRG', 'RBS', 'WeightGain', 'HairGrowth', 'SkinDarkening', 'HairLoss', 'Pimples', 'FastFood', 'RegExercise', 'BP_Systolic', 'BP_Diastolic', 'Follicle_L', 'Follicle_R', 'FSize_L', 'FSize_R', 'Endometrium']

Dtype fixes applied.
  AMH    — missing after coercion: 1
  BetaHCG_II — missing after coercion: 0

Cycle_RI recoded:
Cycle_RI
0    390
1    151
Name: count, dtype: int64

FSH > 500 → NaN: 1 row(s)
LH > 500 → NaN: 1 row(s)
VitD3 > 200 → NaN: 2 row(s)
BP_Systolic < 50 → NaN: 1 row(s)
Pulse_bpm < 25 → NaN: 2 row(s)

Missing values before